# 03 - Carga de datos en Snowflake
## Contratos Menores - Ayuntamiento de Vilagarcía de Arousa

Este notebook carga el CSV limpio generado en el paso anterior en una tabla de Snowflake.

**Entrada:** `data/processed/contratos_limpios.csv`  
**Destino:** tabla `FACT_CONTRATOS` en Snowflake

Las credenciales se leen del fichero `.env` — nunca se escriben directamente en el código.

## 0 — Instalar dependencias
Ejecuta solo la primera vez.

In [13]:
%pip install --prefer-binary -q snowflake-connector-python python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


## 1 — Importaciones y carga de credenciales

In [14]:
import os
import pandas as pd
import snowflake.connector
from pathlib import Path
from dotenv import load_dotenv

# Cargamos las variables del fichero .env en el entorno del proceso
load_dotenv(Path("../.env"))

# Leemos cada credencial del entorno — si alguna está vacía, lo avisamos
ACCOUNT   = os.getenv("SNOWFLAKE_ACCOUNT")
USER      = os.getenv("SNOWFLAKE_USER")
PASSWORD  = os.getenv("SNOWFLAKE_PASSWORD")
WAREHOUSE = os.getenv("SNOWFLAKE_WAREHOUSE")
DATABASE  = os.getenv("SNOWFLAKE_DATABASE")
SCHEMA    = os.getenv("SNOWFLAKE_SCHEMA")

# Verificamos que todas las variables están rellenas antes de continuar
vacias = [k for k, v in {
    "ACCOUNT": ACCOUNT, "USER": USER, "PASSWORD": PASSWORD,
    "WAREHOUSE": WAREHOUSE, "DATABASE": DATABASE, "SCHEMA": SCHEMA
}.items() if not v]

if vacias:
    raise ValueError(f"Faltan credenciales en .env: {vacias}")

print(f"Cuenta:    {ACCOUNT}")
print(f"Usuario:   {USER}")
print(f"Warehouse: {WAREHOUSE}")
print(f"Base:      {DATABASE}.{SCHEMA}")
print("Credenciales cargadas correctamente.")

Cuenta:    aisfeil-lu45385
Usuario:   socratesagudo
Warehouse: FRAUDE_VILAGARCIA
Base:      FRAUDE_VILAGARCIA.PUBLIC
Credenciales cargadas correctamente.


## 2 — Conectar a Snowflake

In [ ]:
# Abrimos la conexión con Snowflake usando las credenciales del .env
conn = snowflake.connector.connect(
    account   = ACCOUNT,
    user      = USER,
    password  = PASSWORD,
    warehouse = WAREHOUSE,
    database  = DATABASE,
    schema    = SCHEMA,
)

# El cursor es el objeto que ejecuta las sentencias SQL
cur = conn.cursor()

print("Conexión establecida.")
cur.execute("SELECT CURRENT_VERSION()")
version = cur.fetchone()[0]
print(f"Versión de Snowflake: {version}")

## 3 — Crear la tabla `FACT_CONTRATOS`

Si la tabla ya existe la recreamos desde cero (`CREATE OR REPLACE`) para garantizar
que el esquema está siempre actualizado con el CSV más reciente.

In [16]:
# Definimos el esquema de la tabla fact principal
# Los tipos siguen la nomenclatura de Snowflake
DDL = """
CREATE OR REPLACE TABLE FACT_CONTRATOS (
    num_referencia        VARCHAR,        -- identificador único del expediente
    anio_contrato         INTEGER,        -- año extraído del número de referencia
    num_contrato_anio     INTEGER,        -- secuencial dentro del año
    fecha_formalizacion   DATE,           -- fecha real (puede ser nula)
    fecha_estimada        DATE,           -- fecha real o sintética — siempre rellena
    entidad_contratante   VARCHAR,
    tipo_contrato         VARCHAR,        -- Obras / Servicios / Suministros
    tipo_procedimiento    VARCHAR,
    sistema_contratacion  VARCHAR,
    tramitacion           VARCHAR,
    objeto_contrato       VARCHAR,
    codigo_cpv            VARCHAR,
    plazo_meses           FLOAT,
    valor_estimado_eur    FLOAT,
    presupuesto_base_eur  FLOAT,
    importe_sin_iva_eur   FLOAT,
    importe_con_iva_eur   FLOAT,
    flag_limite           VARCHAR,        -- ok / cerca_del_limite / supera_limite
    nif_contratista       VARCHAR,
    nif_valido            BOOLEAN,        -- True si pasa el checksum
    motivo_invalido       VARCHAR,        -- razón si nif_valido = False
    nombre_contratista    VARCHAR,
    contratistas_raw      VARCHAR,        -- campo original para trazabilidad
    observaciones         VARCHAR,
    url_licitacion        VARCHAR
)
"""

cur.execute(DDL)
print("Tabla FACT_CONTRATOS creada correctamente.")

Tabla FACT_CONTRATOS creada correctamente.


## 4 — Cargar el CSV procesado

In [17]:
RUTA_CSV = Path("../data/processed/contratos_limpios.csv")

# Leemos el CSV con los tipos correctos
df = pd.read_csv(
    RUTA_CSV,
    encoding     = "utf-8-sig",   # respeta el BOM que añadimos al guardar
    parse_dates  = ["fecha_formalizacion", "fecha_estimada"],
    dtype        = {"anio_contrato": "Int64", "num_contrato_anio": "Int64"},
)

# Convertimos las columnas datetime a datetime.date nativo de Python.
# El conector de Snowflake no acepta pd.Timestamp; sí acepta datetime.date.
for col in df.select_dtypes(include=["datetime64"]).columns:
    df[col] = df[col].apply(lambda x: x.date() if pd.notna(x) else None)

# Convertimos Int64 (entero nullable de pandas) a int nativo o None.
# El conector tampoco acepta numpy.int64 ni pandas NA.
for col in df.select_dtypes(include=["Int64", "Int32", "Int16", "Int8"]).columns:
    df[col] = df[col].apply(lambda x: None if pd.isna(x) else int(x))

# Reemplazamos los NA restantes por None para que Snowflake los interprete como NULL
df = df.where(pd.notna(df), other=None)

print(f"Filas a cargar: {len(df)}")
df.head(3)

Filas a cargar: 607


,num_referencia,anio_contrato,num_contrato_anio,fecha_formalizacion,fecha_estimada,entidad_contratante,tipo_contrato,tipo_procedimiento,sistema_contratacion,tramitacion,...,importe_sin_iva_eur,importe_con_iva_eur,flag_limite,nif_contratista,nif_valido,motivo_invalido,nombre_contratista,contratistas_raw,observaciones,url_licitacion
0,CT 02/2023,2023,1,None,2023-02-02,Ayuntamiento Vilagarcía de Arousa,Suministros,Contrato menor,No aplica,Ordinaria,...,2394.00,2896.74,ok,B36127405,True,NaN,REDINOR S.L.,B36127405 - REDINOR S.L.,NaN,https://contrataciondelestado.es/wps/poc?uri=d...
1,CT 75/2022,2022,2,None,2022-10-10,Ayuntamiento Vilagarcía de Arousa,Servicios,Contrato menor,No aplica,Ordinaria,...,14849.00,17967.29,cerca_del_limite,35458826W,True,NaN,MANUEL TANOIRA CARBALLO,35458826W - MANUEL TANOIRA CARBALLO,NaN,https://contrataciondelestado.es/wps/poc?uri=d...
2,CT 05/2023,2023,2,None,2023-08-27,Ayuntamiento Vilagarcía de Arousa,Suministros,Contrato menor,No aplica,Ordinaria,...,5559.96,6727.56,ok,B15090061,True,NaN,"TARRIO Y SUAREZ, SL","B15090061 - TARRIO Y SUAREZ, SL",NaN,https://contrataciondelestado.es/wps/poc?uri=d...


In [18]:
# Convertimos cada fila a una tupla y la insertamos en lotes de 1000 filas
# para no superar el límite de tamaño de mensaje de Snowflake
import numpy as np

columnas  = ", ".join(df.columns)
wildcards = ", ".join(["%s"] * len(df.columns))
INSERT_SQL = f"INSERT INTO FACT_CONTRATOS ({columnas}) VALUES ({wildcards})"

# Convierte cualquier tipo que Snowflake no reconoce a su equivalente Python nativo.
# Cubre: NaN/NaT/NA → None, pd.Timestamp → date, numpy int/float/bool → tipos Python.
def normalizar(valor):
    if valor is None:
        return None
    try:
        if pd.isna(valor):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(valor, pd.Timestamp):
        return valor.date()
    # numpy integers (int8, int16, int32, int64, uint*) → int
    if isinstance(valor, np.integer):
        return int(valor)
    # numpy floats → float
    if isinstance(valor, np.floating):
        return float(valor)
    # numpy bool → bool
    if isinstance(valor, np.bool_):
        return bool(valor)
    return valor

BATCH_SIZE = 1000
filas = [
    tuple(normalizar(v) for v in row)
    for row in df.itertuples(index=False, name=None)
]

insertadas = 0
for i in range(0, len(filas), BATCH_SIZE):
    lote = filas[i : i + BATCH_SIZE]
    cur.executemany(INSERT_SQL, lote)
    insertadas += len(lote)
    print(f"  Insertadas {insertadas}/{len(filas)} filas...")

conn.commit()  # confirmamos la transacción
print(f"\nCarga completada: {insertadas} filas insertadas.")

  Insertadas 607/607 filas...

Carga completada: 607 filas insertadas.


## 5 — Verificación de la carga

In [ ]:
# Comprobamos que el número de filas en Snowflake coincide con el CSV
cur.execute("SELECT COUNT(*) FROM FACT_CONTRATOS")
n_snowflake = cur.fetchone()[0]
n_csv       = len(df)

print(f"Filas en CSV:       {n_csv}")
print(f"Filas en Snowflake: {n_snowflake}")
print(f"Coinciden:          {'✓' if n_csv == n_snowflake else '✗ — revisar'}")

In [ ]:
# Verificamos los tipos de dato inferidos por Snowflake
cur.execute("""
    SELECT COLUMN_NAME, DATA_TYPE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_NAME = 'FACT_CONTRATOS'
    ORDER BY ORDINAL_POSITION
""")
tipos = cur.fetchall()

print("Esquema de la tabla en Snowflake:")
for nombre, tipo in tipos:
    print(f"  {nombre:<25} {tipo}")

In [ ]:
# Comprobamos duplicados por número de referencia (debería ser 0)
cur.execute("""
    SELECT num_referencia, COUNT(*) AS n
    FROM FACT_CONTRATOS
    GROUP BY num_referencia
    HAVING COUNT(*) > 1
    ORDER BY n DESC
    LIMIT 10
""")
duplicados = cur.fetchall()

if duplicados:
    print("Referencias duplicadas detectadas:")
    for ref, n in duplicados:
        print(f"  {ref}: {n} veces")
else:
    print("Sin duplicados por número de referencia. ✓")

In [ ]:
# Vista previa de los primeros registros desde Snowflake
cur.execute("""
    SELECT num_referencia, anio_contrato, tipo_contrato,
           presupuesto_base_eur, nombre_contratista, flag_limite
    FROM FACT_CONTRATOS
    LIMIT 5
""")
muestra = cur.fetchall()

print("Muestra de datos en Snowflake:")
cabecera = ["num_referencia", "anio", "tipo", "presupuesto", "contratista", "flag"]
print("  " + "  |  ".join(f"{c:<15}" for c in cabecera))
print("  " + "-" * 90)
for fila in muestra:
    print("  " + "  |  ".join(f"{str(v):<15}" for v in fila))

## 6 — Cerrar la conexión

In [23]:
# Cerramos cursor y conexión para liberar recursos
cur.close()
conn.close()
print("Conexión cerrada.")

Conexión cerrada.
